> ## SOLUTION / ANSWER KEY &mdash; Lab 9.3
> This is the **completed** notebook (all `___` blanks filled). For the student version, open
> [`../lab-03-compute-metrics.ipynb`](../lab-03-compute-metrics.ipynb). Trainer use &mdash; or self-check after you've tried it yourself.

# Lab 9.3 &mdash; Compute Derived Metrics

**Level:** Beginner &nbsp;|&nbsp; **Est. time:** 25 min &nbsp;|&nbsp; **Day 5 &middot; Module 9 &mdash; Agents in Finance, Healthcare &amp; Cybersecurity**

### What you'll do
- Compute year-over-year growth from grounded figures
- Compute a margin as a percentage
- Use a safe calculator -- never bare eval on financial input

> **How this lab works (experiential flow):** read the **Concept**, run the **Demo** to see it work, then complete **Your Turn** by replacing every `___` placeholder. Run the **grader** cell at the end &mdash; it prints `[PASS]` / `[FAIL]` / `[TODO]` and a final `Score`. Aim for a full score.

> **Framework note:** the graded steps are **offline &amp; deterministic** (pure Python stdlib); the agent-assembly labs reuse the **LangChain-shaped shim** from Modules 6&ndash;8. Advanced labs end with an **optional** cell that runs the **real** library. You are building the **financial-report insight agent** &mdash; the client's Lab 5.1. It grounds &amp; cites every figure, gives **no advice**, and has **no trade tool** &mdash; a human analyst decides.

**Reference:** [Module 9 slides &mdash; The financial-report insight agent, end to end](../../presentation/day5-module9-agents-in-industry.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 9 labs](./index.html)

In [ ]:
# Setup -- run me first
import os
WORK = "/tmp/biaa-lab-09-03"
os.makedirs(WORK, exist_ok=True)
print("Working dir:", WORK)

## Concept
The insight agent computes the **derived metrics** an analyst cares about (deck slides 7&ndash;8):
**year-over-year growth**, **margins**, notable movements &mdash; all from the **grounded** figures,
never invented. Financial math must be exact and safe, so compute goes through a small **AST-based
safe calculator**, never bare `eval` on model output.

In [ ]:
import ast, operator
# safe arithmetic: walk a parsed AST of numbers + (+ - * / ** unary-minus) -- never bare eval()
_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
        ast.Div: operator.truediv, ast.USub: operator.neg, ast.Pow: operator.pow}
def safe_calc(expr):
    def ev(node):
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        if isinstance(node, ast.BinOp):
            return _OPS[type(node.op)](ev(node.left), ev(node.right))
        if isinstance(node, ast.UnaryOp):
            return _OPS[type(node.op)](ev(node.operand))
        raise ValueError("unsupported expression")
    return ev(ast.parse(expr, mode="eval").body)

# A small financial report -- each figure carries its SOURCE, so every claim can be cited.
REPORT = {
    "revenue":    {"value": 120.0, "unit": "M", "source": "p4, income stmt"},
    "net_income": {"value": 9.0,   "unit": "M", "source": "p4, income stmt"},
    "total_debt": {"value": 40.0,  "unit": "M", "source": "p7, balance sheet"},
}
PRIOR = {"revenue": 107.1, "net_income": 9.7, "total_debt": 25.0}   # prior period, for YoY

print("safe compute:", safe_calc("(120-107.1)/107.1*100"), "(revenue YoY %)")

## Your Turn
Complete `yoy_growth` (percent change) and `margin_pct` (net income over revenue).

In [ ]:
def yoy_growth(current, prior):
    # percent change year over year, rounded to 1 dp
    return round((current - prior) / prior * 100, 1)

def margin_pct(net_income, revenue):
    # net margin as a percentage, rounded to 1 dp
    return round(net_income / revenue * 100, 1)

try:
    rev_now = REPORT['revenue']['value']; rev_prior = PRIOR['revenue']
    ni_now  = REPORT['net_income']['value']
    print('revenue YoY  :', yoy_growth(rev_now, rev_prior), '%')
    print('net margin   :', margin_pct(ni_now, rev_now), '%')
    print('prior margin :', margin_pct(PRIOR['net_income'], PRIOR['revenue']), '%')
except Exception as e:
    print('Fill the blanks, then re-run.', type(e).__name__)

In [ ]:
# === Auto-grader: run after filling the blanks above ===
_results = []
def _rec(label, status, extra=""):
    _results.append(status); print(f"[{status}] {label}" + (f" -- {extra}" if extra else ""))
def expect(label, got, want):
    if got == "___" or got is None: _rec(label, "TODO")
    elif got == want: _rec(label, "PASS")
    else: _rec(label, "FAIL", f"got {got!r}")
def expect_true(label, fn):
    try: _rec(label, "PASS" if fn() else "FAIL")
    except Exception as e: _rec(label, "TODO", type(e).__name__)

expect_true("revenue YoY is +12.0%", lambda: yoy_growth(120.0, 107.1) == 12.0)
expect_true("a decline gives a negative growth", lambda: yoy_growth(90.0, 100.0) == -10.0)
expect_true("net margin is 7.5%", lambda: margin_pct(9.0, 120.0) == 7.5)
expect_true("the prior margin is ~9.1%", lambda: margin_pct(9.7, 107.1) == 9.1)
expect_true("metrics are grounded in the passed figures (deterministic)", lambda: yoy_growth(120.0, 107.1) == yoy_growth(REPORT["revenue"]["value"], PRIOR["revenue"]))

_p = _results.count("PASS")
print(f"\nScore: {_p}/{len(_results)}")
print("All checks passed -- lab complete!" if _p == len(_results) else "Keep going: fill the blanks marked ___ and re-run.")

---
### Key takeaway
Derived metrics come from the grounded figures, computed exactly through a safe calculator. The margin fell from 9.1% to 7.5% even as revenue grew -- exactly the kind of movement the agent must surface next.

[&#8592; All Module 9 labs](./index.html) &nbsp;&middot;&nbsp; [Module 9 slides](../../presentation/day5-module9-agents-in-industry.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>